# 03 — Structured ensemble (D8) · axes B / C / D · **gates G7 + G12**

Third of four (spec §9, step 3). Axis B = band cutoff × {0.5, 1, 2}; axis C = leave-one-out
**by name** (all parts of a name dropped together, `no_link` parts included — independent in the
graph but sharing the name's realisation risk); axis D = β ∈ {1.5, 2.5, 4.0}. One cached CWD
set serves every member; serial with memmaps on purpose. Resumable: a member is done when its
`member.json` exists.

**Axis C is the one to read closely.** The decision after this notebook — whether to open
Phase 7 (climate-modified routing) at all — is made on axis-C stability; it does NOT block
notebook 04, which is climate-free by construction (D14).


In [1]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/northern_connectivity/, below the repo root where config.py
# and the corridor engine modules (corridors_prep / corridor_graph / corridors_core /
# corridors_ensemble) sit. Same bootstrap pattern as analyses/y2y/.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
import corridors_ensemble as ce
for _m in (config, cp, cg, cc, ce):
    importlib.reload(_m)

KEY = "north"


In [2]:
# ---- Re-attach to the run created by 02_calibrate_baseline ------------------
# One run spans notebooks 02-04: 02 created it via cc.start(); here cc.load() reads ONLY the run
# dir (run_config.json + the H7 copies), then the cached CWD + graph are re-derived -- the CWD
# stage is a cache HIT (seconds), the network rebuild is minutes. cwd_cutoff_abs was written into
# run_config.json by cc.set_cutoff in notebook 02, so nothing here depends on config.py.
RUN = "v2_run002"                       # <-- the run to continue

A = cc.load(config.RESULTS_DIR / "corridors_north" / RUN)
cc.resistance(A)
cc.cost_distances(A)                    # cache HIT
cc.corridor_network(A, verbose=False)
A


Northern BC + Yukon: routing grid 2809x5767 @ 300 m = 9,696,945 routable cells (872,725 km²)
  cropped from the 4285x5767 warped window (66% of its cells) = anchors + 100 km routing buffer
  merged node: IPCA · Peel Watershed - SMA/WA absorbs Teetł’it Gwinjik (Peel River) (4,147 km² nested)
  merged node: PA · Nj ‘Iinlii” Jjik (Fishing Branch) Habitat Protection Area absorbs Fishing Branch Wilderness Preserve (5,355 km² nested)
  merged node: PA · Neah Conservancy absorbs Ne'ah – Horseranch Range Deadwood Lake Protected Area (2,312 km² nested)
nodes: 42 (10 IPCAs + 32 existing PAs >= 200 km²)  [min node size 25 km² = 278 cells @ 300 m]
  dropped 2 IPCA(s) below 25 km² in region: Wëdzey Nähuzhi (Matson Uplands), Łuk Tthe K’ät (Scottie Creek Wetlands)
  node land: 242,540 km² (excluded from the corridor)
D16 parts: 42 names -> 48 seed parts -> 42 routing units  (4 multipart: Dene Kʼéh Kusān 3p/link_locked; Liard River Corridor Park 3p/link_locked; Nahanni National Park Reserve Of Canada 

<corridors v2_run002 | 42 nodes | 2809x5767 | 58 edges | corridor 33,041 km²>

## Gates — **G7** (cache reuse is sound) and **G12** (47 distinct members by config-hash)


In [3]:
ce.gate_g7(A)
ce.gate_g12(A)


G7 cache reuse: drop-nothing member vs baseline Jaccard = 1.000000, edges 52 vs 52 unit edges (+6 locked, shared)
  G7 OK
  design: 2 duplicate member(s) removed by config-hash (cutoff x1, beta 2.5)
G12 OK: 47 distinct members (baseline + 2 B + 42 C + 2 D), duplicates removed by config-hash


True

## Run every member (resumable)


In [4]:
ce.run(A)


  design: 2 duplicate member(s) removed by config-hash (cutoff x1, beta 2.5)
ensemble: 47 members | 0 already done | 47 to solve
  [  1/47] run_0000 baseline   baseline                             33,041 km²
  [  2/47] run_0001 B_cutoff   cutoff x0.5                          19,039 km²
  [  3/47] run_0002 B_cutoff   cutoff x2                            57,190 km²
  [  4/47] run_0003 D_beta     beta 1.5                             26,510 km²
  [  5/47] run_0004 D_beta     beta 4                               34,341 km²
  [  6/47] run_0005 C_loo      drop Peel Watershed                  31,420 km²
  [  7/47] run_0006 C_loo      drop Dene Kʼéh Kusān                 37,412 km²
  [  8/47] run_0007 C_loo      drop Tū Łī́dlini                     38,421 km²
  [  9/47] run_0008 C_loo      drop T’akú Tlatsini                  31,218 km²
  [ 10/47] run_0009 C_loo      drop Tahltan                         33,041 km²
  [ 11/47] run_0010 C_loo      drop Wilps Gwininitxw                35,891 km²
  

PosixPath('/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/output_data/corridors_north/v2_run002/ensemble')

## Collect — attribution surface + per-axis rasters (**D15**)

Written as `ensemble_attribution.tif` (+ `attribution_<axis>.tif`), used for the robust-core
threshold (0.9) and attribution only. NEVER labelled "frequency": ~42 of 47 members are
leave-one-out, so the fraction is "share of dropped names that didn't matter", not a
near-optimal sampling frequency — that product is notebook 04's near-optimality surface.


In [5]:
ce.collect(A)


ensemble over 47 members
  robust core (attribution >= 0.9): 32,598 km² of 95,844 km² ever used

  per-axis attribution (1 - Jaccard vs baseline = how much that axis moves the network):
    B_cutoff   n=  2  mean 0.423  max 0.424  (cutoff x0.5)
    C_loo      n= 42  mean 0.081  max 0.317  (drop Nahanni)
    D_beta     n=  2  mean 0.177  max 0.272  (beta 1.5)

  most structurally load-bearing names (lowest Jaccard when dropped) -- AXIS C IS THE ONE TO READ CLOSELY:
    PA · Nahanni National Park Reserve Of Canada         J=0.683
    IPCA · Dene Kʼéh Kusān                               J=0.710
    PA · Liard River Corridor Park                       J=0.792
    IPCA · Tū Łī́dlini (Ross River)                      J=0.808
    PA · Pine Le Moray Park                              J=0.814  ** NETWORK SPLITS INTO 2 **

  wrote ensemble_attribution.tif, attribution_<axis>.tif, members.csv, edge_frequency.csv -> output_data/corridors_north/v2_run002/ensemble


<corridors v2_run002 | 42 nodes | 2809x5767 | 58 edges | corridor 33,041 km²>